In [22]:
import bm25s
import pandas as pd
from sklearn.preprocessing import normalize
from scipy.sparse import save_npz
from tqdm import tqdm
import nltk
from scipy.sparse import load_npz
tqdm.pandas()

In [6]:
df = pd.read_parquet("../../data/datasets/df_completo.parquet")

In [28]:
import bm25s

# Create your corpus here
corpus = [
    "a cat is a feline and likes to purr",
    "a dog is the human's best friend and loves to play",
    "a bird is a beautiful animal that can fly",
]

# Tokenize the corpus and index it
corpus_tokens = bm25s.tokenize(corpus)
retriever = bm25s.BM25(corpus=corpus)
retriever.index(corpus_tokens)

# You can now search the corpus with a query
query = "does the fish purr like a cat?"
query_tokens = bm25s.tokenize(query)
docs, scores = retriever.retrieve(query_tokens, k=2)
print(f"Best result (score: {scores[0, 0]:.2f}): {docs[0, 0]}")
corpus_tokens.ids

Split strings:   0%|          | 0/3 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/3 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/3 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Best result (score: 0.86): a cat is a feline and likes to purr


[[0, 1, 2, 3], [4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14]]

## Vectorizer

In [17]:
import numpy as np
import bm25s
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin

class BM25LVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, k1=1.5, b=0.75, delta=0.5, norm="l2"):
        self.k1 = k1
        self.b = b
        self.delta = delta
        self.norm = norm
        self.retriever = None
        self.vocabulary_ = None
        self._old_to_new_cols = None  # Mapeamento para reordenar as colunas no transform
        
    def fit(self, X, y=None):
        """
        Aprende o vocabulário e calcula as estatísticas do BM25L sobre o corpus.
        """
        # Força tokenização limpa por split (removendo strings vazias acidentais)
        corpus_tokens = [[w for w in doc.split() if w != ''] for doc in X]
        
        # Inicializa e indexa o retriever do bm25s
        self.retriever = bm25s.BM25(
            method="bm25l",
            k1=self.k1,
            b=self.b,
            delta=self.delta,
            corpus=corpus_tokens
        )
        self.retriever.index(corpus_tokens, show_progress=False)
        
        # 1. Recupera o dicionário de vocabulário original da biblioteca
        vocab_original = {term: idx for term, idx in self.retriever.vocab_dict.items() if term != ''}

        # 2. Cria o novo vocabulário ordenado alfabeticamente (Estilo Sklearn)
        termos_ordenados = sorted(vocab_original.keys())
        self.vocabulary_ = {term: idx for idx, term in enumerate(termos_ordenados)}
        
        # 3. Cria um array mapeando qual coluna nova alfabética corresponde a cada coluna original da lib
        # Ex: se o termo na coluna original 0 deve ir para a coluna nova 5, o array na posição 0 terá o valor 5
        self._old_to_new_cols = np.array([self.vocabulary_[term] for term in vocab_original.keys()])
        
        return self

    def transform(self, X):
        """
        Transforma os documentos na matriz esparsa CSR do sklearn (Documentos x Vocabulário Ordenado)
        com os scores originais reconstruídos.
        """
        if self.retriever is None:
            raise RuntimeError("O vetorizador precisa ser ajustado com 'fit' antes de transformar.")
            
        data = self.retriever.scores["data"]
        indices = self.retriever.scores["indices"]
        indptr = self.retriever.scores["indptr"]
        nonoccurrence_array = self.retriever.nonoccurrence_array
        
        num_docs = self.retriever.scores["num_docs"]
        num_terms = len(self.vocabulary_)

        # 1. Reconstrói os scores originais (Reversão do Shifting)
        term_indices = np.zeros_like(data, dtype=np.int32)
        for i in range(len(indptr) - 1):
            start, end = indptr[i], indptr[i+1]
            term_indices[start:end] = i

        scores_originais = data + nonoccurrence_array[term_indices]

        # 2. Monta a matriz bruta respeitando o indptr da biblioteca (Termos x Documentos)
        # Isso evita o erro de ValueError: index pointer size
        matriz_raw = csr_matrix((scores_originais, indices, indptr), shape=(num_terms, num_docs))
        
        # 3. Transpõe (.T) para virar a estrutura padrão do sklearn (Documentos x Termos)
        matriz_csr = matriz_raw.T.tocsr()
        
        # 4. FIX: Reordena as colunas de acordo com a ordem alfabética do vocabulary_
        # np.argsort nos dá os índices necessários para ordenar as colunas corretamente
        sorting_order = np.argsort(self._old_to_new_cols)
        matriz_csr = matriz_csr[:, sorting_order]
        if self.norm is not None:
            # axis=1 garante que a normalização aconteça por linha (documento)
            # copy=False altera os dados in-place para máxima eficiência de memória
            matriz_csr = normalize(matriz_csr, norm=self.norm, axis=1, copy=False)
        return matriz_csr

    def fit_transform(self, X, y=None):
        """
        Ajusta aos dados e retorna a matriz de termos modificada.
        """
        return self.fit(X, y).transform(X)

## Pré-processamento
* Fonte: https://github.com/Convenio-Camara-dos-Deputados/papers-JURIX-2021

In [18]:
class Savoy:

    def __removeAllPTAccent(self, old_word):
        word = list(old_word)
        len_word = len(word)-1
        for i in range(len_word, -1, -1):
            if word[i] == 'ä':
                word[i] = 'a'
            if word[i] == 'â':
                word[i] = 'a'
            if word[i] == 'à':
                word[i] = 'a'
            if word[i] == 'á':
                word[i] = 'a'
            if word[i] == 'ã':
                word[i] = 'a'
            if word[i] == 'ê':
                word[i] = 'e'
            if word[i] == 'é':
                word[i] = 'e'
            if word[i] == 'è':
                word[i] = 'e'
            if word[i] == 'ë':
                word[i] = 'e'
            if word[i] == 'ï':
                word[i] = 'i'
            if word[i] == 'î':
                word[i] = 'i'
            if word[i] == 'ì':
                word[i] = 'i'
            if word[i] == 'í':
                word[i] = 'i'
            if word[i] == 'ü':
                word[i] = 'u'
            if word[i] == 'ú':
                word[i] = 'u'
            if word[i] == 'ù':
                word[i] = 'u'
            if word[i] == 'û':
                word[i] = 'u'
            if word[i] == 'ô':
                word[i] = 'o'
            if word[i] == 'ö':
                word[i] = 'o'
            if word[i] == 'ó':
                word[i] = 'o'
            if word[i] == 'ò':
                word[i] = 'o'
            if word[i] == 'õ':
                word[i] = 'o'
            if word[i] == 'ç':
                word[i] = 'c'

        new_word = "".join(word)
        return new_word

    def __finalVowelPortuguese(self, word):
        len_word = len(word)
        if len_word > 3:
            if word[-1] == 'e' or word[-1] == 'a' or word[-1] == 'o':
                word = word[:-1]

        return word

    def __remove_PTsuffix(self, word):
        len_word = len(word)

        if len_word > 3:
            if word[-1] == 's' and word[-2] == 'e' and (word[-3] == 'r' or word[-3] == 's' or word[-3] == 'z' or word[-3] == 'l'):
                word = word[:-2]
                return word
        if len_word > 2:
            if word[-1] == 's' and word[-2] == 'n':
                new_word = list(word)
                new_word[-2] = 'm'
                sing = "".join(new_word)
                sing = sing[:-1]
                return sing

        if len_word > 3:
            if (word[-1] == 's' and word[-2] == 'i') and (word[-3] == 'e' or word[-3] == 'é'):
                new_word = list(word)
                new_word[-3] = 'e'
                new_word[-2] = 'l'
                sing = "".join(new_word)
                sing = sing[:-1]
                return sing

        if len_word > 3:
            if word[-1] == 's' and word[-2] == 'i' and word[-3] == 'a':
                new_word = list(word)
                new_word[-2] = 'l'
                sing = "".join(new_word)
                sing = sing[:-1]
                return sing

        if len_word > 3:
            if word[-1] == 's' and word[-2] == 'i' and word[-3] == 'ó':
                new_word = list(word)
                new_word[-3] = 'o'
                new_word[-2] = 'l'
                sing = "".join(new_word)
                sing = sing[:-1]
                return sing

        if len_word > 3:
            if word[-1] == 's' and word[-2] == 'i':
                new_word = list(word)
                new_word[-1] = 'l'
                sing = "".join(new_word)
                return sing

        if len_word > 2:
            if word[-1] == 's' and word[-2] == 'e' and word[-3] == 'õ':
                new_word = list(word)
                new_word[-3] = 'ã'
                new_word[-2] = 'o'
                sing = "".join(new_word)
                sing = sing[:-1]
                return sing
            if word[-1] == 's' and word[-2] == 'e' and word[-3] == 'ã':
                new_word = list(word)
                new_word[-2] = 'o'
                sing = "".join(new_word)
                sing = sing[:-1]
                return sing

        if len_word > 5:
            if word[-1] == 'e' and word[-2] == 't' and word[-3] == 'n' and word[-4] == 'e' and word[-5] == 'm':
                word = word[:-5]
                return word

        if len_word > 2:
            if word[-1] == 's':
                word = word[:-1]

        return word

    def __normFemininPortuguese(self, word):

        len_word = len(word)

        if len_word < 3 or word[-1] != 'a':
            return word

        if len_word > 6:

            if word[-2] == 'h' and word[-3] == 'n' and word[-4] == 'i':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 'c' and word[-3] == 'a' and word[-4] == 'i':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 'r' and word[-3] == 'i' and word[-4] == 'e':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

        if len_word > 5:
            if word[-2] == 'n' and word[-3] == 'o':
                new_word = list(word)
                new_word[-3] = 'ã'
                new_word[-2] = 'o'
                masc = "".join(new_word)
                masc = masc[:-1]
                return masc

            if word[-2] == 'r' and word[-3] == 'o':
                word = word[:-1]
                return word

            if word[-2] == 's' and word[-3] == 'o':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 's' and word[-3] == 'e':
                new_word = list(word)
                new_word[-3] = 'ê'
                masc = "".join(new_word)
                masc = masc[:-1]
                return masc

            if word[-2] == 'c' and word[-3] == 'i':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 'd' and word[-3] == 'i':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 'd' and word[-3] == 'a':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 'v' and word[-3] == 'i':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 'm' and word[-3] == 'a':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

            if word[-2] == 'n':
                new_word = list(word)
                new_word[-1] = 'o'
                masc = "".join(new_word)
                return masc

        return word

    def stem(self, word):
        len_word = len(word)
        if len_word > 2:
            word = self.__remove_PTsuffix(word)
            word = self.__normFemininPortuguese(word)
            word = self.__finalVowelPortuguese(word)
            word = self.__removeAllPTAccent(word)

        return word

In [19]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('rslp')

from nltk.tokenize import word_tokenize
from string import punctuation
import nltk
from unicodedata import normalize as normalize_text
from nltk.util import ngrams
from nltk.tokenize import RegexpTokenizer

# Remove os acentos de uma string
def _remove_acentos(txt):
    return normalize_text('NFKD', txt).encode('ASCII', 'ignore').decode('ASCII')

def stem_Savoy(txt):
    stemmer = Savoy()
    terms = word_tokenize(txt)
    terms = [stemmer.stem(word) for word in terms]
    return terms


# Remoção de stopwords + acentuação + steming + pontuação
def preprocess_Savoy(txt):
    txt = _remove_acentos(txt)
    stopwords = nltk.corpus.stopwords.words("portuguese")
    stopwords.extend(list(punctuation))

    stemmer = Savoy()
    tokenizer = RegexpTokenizer('\w+')
    terms = tokenizer.tokenize(txt.lower())
    terms = [stemmer.stem(word) for word in terms if word not in stopwords]
    return terms

#---------------------------------------------------------------------------------

def n_gram(txt, n):
    terms = word_tokenize(txt)
    ngram = list(ngrams(terms, n))

    return ngram

def preprocess_ngram_savoy(txt, n):
    txt = _remove_acentos(txt)
    stopwords = nltk.corpus.stopwords.words("portuguese")
    stopwords.extend(list(punctuation))

    stemmer = Savoy()
    # terms = word_tokenize(txt.lower())
    tokenizer = RegexpTokenizer('\w+')
    terms = tokenizer.tokenize(txt.lower())
    terms = [stemmer.stem(word) for word in terms if word not in stopwords]

    ngram = list(ngrams(terms, n))

    return ngram

<>:32: SyntaxWarning: invalid escape sequence '\w'
<>:52: SyntaxWarning: invalid escape sequence '\w'
<>:32: SyntaxWarning: invalid escape sequence '\w'
<>:52: SyntaxWarning: invalid escape sequence '\w'
C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_21572\1843011346.py:32: SyntaxWarning: invalid escape sequence '\w'
  tokenizer = RegexpTokenizer('\w+')
C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_21572\1843011346.py:52: SyntaxWarning: invalid escape sequence '\w'
  tokenizer = RegexpTokenizer('\w+')
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\3675-Robson\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\3675-Robson\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\3675-Robson\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!


In [20]:
def preprocess_txt_bm25(txt):

    unigrams = preprocess_Savoy(txt)

    bigrams = preprocess_ngram_savoy(
        txt,
        2
    )

    bigrams = [
        "_".join(bg)
        for bg in bigrams
    ]

    return " ".join(
        unigrams + bigrams
    )

## Matrizes documento x documento (Scores para retrieval)

In [ ]:
def build_score_matrix_bm25s(corpus):
    corpus_tokens = bm25s.tokenize(corpus, show_progress=False)
    
    retriever = bm25s.BM25(method="bm25l")
    retriever.index(corpus_tokens)
    
    n = len(corpus)
    score_matrix = np.empty((n, n), dtype=np.float32)
    
    for i in tqdm(range(n), desc="Calculando Matriz BM25S", leave=False):

      
        score_matrix[i] = retriever.get_scores(corpus_tokens.ids[i])
        
    # Desconsidera o auto-match na similaridade do retrieval
    np.fill_diagonal(score_matrix, -np.inf)
    
    return score_matrix





materias = [
    "PEC_6_2019",
    "MPV_612_2013",
    "PLP_68_2024"
]

campos_textuais = [
    "texto",
    "texto_preprocessado",
    "texto_preprocessado_sem_justificativa"
]

resultados = {}
resultados_df = {}

# OBS: Certifique-se de que o DataFrame global 'embeddings' e a lista 
# 'embedding_cols' de colunas densas estejam instanciados no seu ambiente antes de rodar.

for materia in materias:
    print(f"\n{'='*60}\nProcessando Proposição: {materia}\n{'='*60}")
    
    embeddings_materia = (
        embeddings[embeddings["materia"] == materia]
        .copy()
        .reset_index(drop=True)
    )
    
    niveis = ["tema"]
    if materia == "PLP_68_2024":
        niveis = ["tema_macro", "tema_nivel_2", "tema"]
        
    resultados[materia] = []
    
    # -------------------------------------------------
    # Matrizes de Similaridade BM25L
    # -------------------------------------------------
    bm25_models = {}
    for campo in campos_textuais:
        print(f"-> Computando representações BM25L para campo: '{campo}'")
        
        corpus_raw = (
            embeddings_materia[campo]
            .fillna("")
            .astype(str)
            .str.lower()
            .tolist()
        )
        
        # Geração da matriz bruta (raw)
        bm25_models[f"{campo}__raw"] = build_score_matrix_bm25s(corpus_raw)
        
        # Geração da matriz estruturada (Savoy + Bigrams)
        print(f"   Aplicando Stemming Savoy e N-Grams em '{campo}'...")
        corpus_preprocess = [
            preprocess_txt_bm25(doc)
            for doc in corpus_raw
        ]
        bm25_models[f"{campo}__preprocess"] = build_score_matrix_bm25s(corpus_preprocess)
        
    # -------------------------------------------------
    # Execução das Métricas por Nível de Granularidade
    # -------------------------------------------------
    for nivel in niveis:
        labels = embeddings_materia[nivel].values
        print(f"\nAvaliando Granularidade: [Nível: {nivel}] | Amostras: {len(labels)}")
        
        # Evaluator 1: Embeddings Densos das LLMs
        for col in tqdm(embedding_cols, desc="Avaliando Vetores Densos"):
            X = np.vstack(embeddings_materia[col].values)
            metrics = retrieval_metrics_embeddings(X, labels)
            
            resultados[materia].append({
                "nivel": nivel,
                "embedding": col,
                **metrics
            })
            
        # Evaluator 2: Matrizes Léxicas do BM25L
        for nome, score_matrix in bm25_models.items():
            metrics = retrieval_metrics_from_scores(score_matrix, labels)
            
            resultados[materia].append({
                "nivel": nivel,
                "embedding": f"retrieval__bm25l__{nome}",
                **metrics
            })
            
    # Consolidação e ordenação do ranking da matéria
    resultados_df[materia] = (
        pd.DataFrame(resultados[materia])
        .sort_values(
            ["map", "r_precision", "precision@1"],
            ascending=False
        )
        .reset_index(drop=True)
    )
    
    # Exibe o Top 20 imediato na tela
    print(f"\n>>> TOP 20 RANKING - {materia} <<<")
    display(resultados_df[materia].head(20))
    
    # Limpeza de memória da iteração
    del bm25_models
    gc.collect()

In [27]:
import os
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
import bm25s

# =====================================================
# FUNÇÃO DE GERAÇÃO DA MATRIZ BM25S
# =====================================================
def build_score_matrix_bm25s(corpus):
    corpus_tokens = bm25s.tokenize(corpus, show_progress=False)
    
    retriever = bm25s.BM25(method="bm25l")
    retriever.index(corpus_tokens)
    
    n = len(corpus)
    score_matrix = np.empty((n, n), dtype=np.float32)
    
    for i in tqdm(range(n), desc="Calculando Matriz BM25S", leave=False):
        score_matrix[i] = retriever.get_scores(corpus_tokens.ids[i])
        
    # Desconsidera o auto-match na similaridade do retrieval
    np.fill_diagonal(score_matrix, -np.inf)
    
    return score_matrix


materias = ["PEC_6_2019", "MPV_612_2013", "PLP_68_2024"]
campos_textuais = ["texto", "texto_preprocessado", "texto_preprocessado_sem_justificativa"]

for materia in materias:
    print(f"\n{'='*60}\nGerando matrizes de retrieval para: {materia}\n{'='*60}")
    
    # Filtra os dados da matéria a partir do DataFrame global 'embeddings'
    df_materia = df[df["materia"] == materia].copy().reset_index(drop=True)
    
    for campo in campos_textuais:
        print(f"\n-> Processando campo: '{campo}'")
        
        # 1. Recupera os textos puros (RAW)
        corpus_raw = df_materia[campo].fillna("").astype(str).str.lower().tolist()
        
        # 2. Gera e salva a matriz para RAW
        print("   Gerando matriz RAW...")
        matrix_raw = build_score_matrix_bm25s(corpus_raw)
        np.save(f"../../data/bm25l/doc_doc_scores/sim_matrix_{materia}_{campo}_raw.npy", matrix_raw)
        
        # 3. Limpa memória
        del matrix_raw
        gc.collect()
        
        # 4. Gera e salva a matriz estruturada (PREPROCESS - Savoy + Bigrams)
        print("   Aplicando Stemming Savoy e N-Grams...")
        corpus_preprocess = [preprocess_txt_bm25(doc) for doc in corpus_raw]
        
        print("   Gerando matriz PREPROCESS...")
        matrix_preprocess = build_score_matrix_bm25s(corpus_preprocess)
        np.save(f"../../data/bm25l/doc_doc_scores/sim_matrix_{materia}_{campo}_preprocess.npy", matrix_preprocess)
        
        # 5. Limpa memória
        del matrix_preprocess, corpus_preprocess, corpus_raw
        gc.collect()


Gerando matrizes de retrieval para: PEC_6_2019

-> Processando campo: 'texto'
   Gerando matriz RAW...


BM25S Count Tokens:   0%|          | 0/268 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/268 [00:00<?, ?it/s]

   Aplicando Stemming Savoy e N-Grams...
   Gerando matriz PREPROCESS...


BM25S Count Tokens:   0%|          | 0/268 [00:00<?, ?it/s]

...

BM25S Count Tokens:   0%|          | 0/1974 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1974 [00:00<?, ?it/s]

   Aplicando Stemming Savoy e N-Grams...
   Gerando matriz PREPROCESS...


BM25S Count Tokens:   0%|          | 0/1974 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1974 [00:00<?, ?it/s]

## Matrizes documento x termos/vocabulário (Clustering)

In [24]:
import importlib
import bm25s.scoring
import bm25s

importlib.reload(bm25s.scoring)
importlib.reload(bm25s)

for col in [
    "texto",
    "texto_preprocessado",
    "texto_preprocessado_sem_justificativa"
]:
    print(f"Vetorizando {col}...")
    vectorizer = BM25LVectorizer()

    X = vectorizer.fit_transform(df[col].fillna(""))

    save_npz(f"../../data/bm25l/doc_term_matrices/bm25l_{col}_raw.npz", X)

Vetorizando texto...
Vetorizando texto_preprocessado...
Vetorizando texto_preprocessado_sem_justificativa...


In [26]:
import importlib
import bm25s.scoring
import bm25s

from scipy.sparse import save_npz
from tqdm import tqdm

importlib.reload(bm25s.scoring)
importlib.reload(bm25s)

for col in [
    "texto",
    "texto_preprocessado",
    "texto_preprocessado_sem_justificativa"
]:
    print(f"Pré-processando {col}...")

    corpus = (
        df[col]
        .fillna("")
        .progress_apply(preprocess_txt_bm25)
        .tolist()
    )

    print(f"Vetorizando {col}...")

    vectorizer = BM25LVectorizer()

    X = vectorizer.fit_transform(corpus)

    save_npz(
        f"../../data/bm25l/doc_term_matrices/bm25l_{col}_preprocess.npz",
        X
    )

Pré-processando texto...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2462/2462 [00:44<00:00, 54.98it/s]


Vetorizando texto...
Pré-processando texto_preprocessado...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2462/2462 [00:38<00:00, 63.72it/s]


Vetorizando texto_preprocessado...
Pré-processando texto_preprocessado_sem_justificativa...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2462/2462 [00:16<00:00, 146.38it/s]


Vetorizando texto_preprocessado_sem_justificativa...
